# 🗺️ TaaSim-Casablanca · Mapping & Map-Matching Pipeline
## Notebook 03 — Porto → Casablanca Road-Fidelity Mapping

| Attribute | Value |
|---|---|
| **Version** | 2.0 — Advance mapping |
| **Python** | ≥ 3.11 |
| **Spark** | 3.5.x |
| **RAM** | 8 GB |

---

### What this notebook does

This notebook implements the **real Porto → Casablanca mapping** pipeline. Each real
Porto taxi trip trajectory is **geometrically transformed** into Casablanca's road
network so that the Casablanca trip follows **actual streets**.

### The mapping algorithm 

```
For each Porto trip polyline [[lon,lat], …]:

  1. Extract START = first point, END = last point
  2. Compute relative position within Porto bbox:
       rel_lon = (lon − porto_min_lon) / (porto_max_lon − porto_min_lon)
       rel_lat = (lat − porto_min_lat) / (porto_max_lat − porto_min_lat)
  3. Map to Casablanca bbox:
       casa_lon = casa_min_lon + rel_lon × (casa_max_lon − casa_min_lon)
       casa_lat = casa_min_lat + rel_lat × (casa_max_lat − casa_min_lat)
  4. Snap to nearest Casablanca OSM road nodes
  5. Shortest path on UNDIRECTED graph (avoids one-way blocking)
  6. Extract FULL road geometry (including edge curves)
```

### Pipeline flow

```
§0 Imports  →  §1 Config + Spark  →  §2 Load Porto + Casa graphs
                                              ↓
§3 Compute affine transform  →  §4 Map trips (START/END → road path)
                                              ↓
§5 Enrich: zones, H3, fare, duration  →  §6 Parquet write
                                              ↓
§7 Before/After comparison  →  §8 Validation panels  →  §9 Folium maps
```

### Key innovation vs notebook 02

| Feature | Notebook 02 | This notebook (03) |
|---|---|---|
| **Trip origin** | Random bbox sampling | Actual Porto GPS → affine transform |
| **Trip shape** | Gravity OD, random routes | Porto trajectory shape preserved |
| **Road following** | Dijkstra on directed graph | Dijkstra on **undirected** graph (no one-way blocking) |
| **Geometry** | Node-to-node straight lines | **Full edge geometry** with road curves |
| **Fidelity** | Synthetic urban mobility model | Direct port of real GPS tracks |

## ADR · Architecture Decision Records

### ADR-01 — Relative-position affine mapping 
**Context:** Porto and Casablanca are different-sized cities with different road  
topologies. A simple coordinate offset would place trips outside Casablanca.  
**Decision:** Normalise each Porto coordinate to a [0,1] relative position  
within Porto's road-network bbox, then scale to Casablanca's road bbox.  
**Consequence:** Trips at 30% across Porto map to 30% across Casablanca.

---

### ADR-02 — Undirected graph for routing 
**Context:** Casablanca has many one-way streets. Directed-graph routing  
(`nx.shortest_path` on `MultiDiGraph`) fails for ~30% of O/D pairs.  
**Decision:** Build `G.to_undirected()` and route on that.  
**Consequence:** Route success rate ≥ 90%; geometry is extracted from the  
original directed graph so edge curves are preserved correctly.

---

### ADR-03 — Full edge-geometry extraction
**Context:** `nx.shortest_path` returns node IDs, not coordinates. Using  
just `(G.nodes[n]['x'], G.nodes[n]['y'])` produces straight-segment routes  
that cut through buildings on curved roads.  
**Decision:** `extract_route_geometry()` reads each edge's `geometry`  
attribute (a Shapely LineString) and concatenates the coordinate sequences.  
**Consequence:** Routes follow actual road curves — **exact-fidelity mapping**.

---

### ADR-04 — Parallel routing with ProcessPoolExecutor(2)
**Context:** Each worker loads Porto+Casa graphs → ~2.5 GB per worker.  
**Decision:** Hard-cap `max_workers=2` for 8 GB RAM machines.  
**Consequence:** ~1500 trips/min throughput.

---

### ADR-05 — Both Porto AND Casablanca graphs required
**Context:** The affine transform needs Porto's bbox (derived from its road graph).  
**Decision:** Download and cache both `Porto, Portugal` and `Casablanca, Morocco` drive graphs.  
**Consequence:** ~2× disk cache but enables exact relative-position mapping.

## §0 · Environment Setup & Imports

In [1]:
# %pip install osmnx geopandas shapely folium networkx matplotlib pyarrow pyspark pytz h3 scipy

import gc, json, math, os, time, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import geopandas as gpd
import numpy as np
import pandas as pd
import pytz
from shapely.geometry import Point, Polygon, LineString

import networkx as nx
import osmnx as ox

try:
    import h3
    H3_AVAILABLE = True
    print(f"H3 version: {h3.__version__}")
except ImportError:
    H3_AVAILABLE = False
    print(" h3 not installed — hex columns skipped")

import folium
import matplotlib.pyplot as plt
from folium.plugins import HeatMap

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

warnings.filterwarnings("ignore")
ox.settings.log_console = False
ox.settings.use_cache   = True

RNG = np.random.default_rng(42)
print(f" Imports OK | numpy {np.__version__} | osmnx {ox.__version__} | networkx {nx.__version__}")

H3 version: 4.4.2
 Imports OK | numpy 1.26.4 | osmnx 2.1.0 | networkx 3.2.1


## §1 · Configuration & Spark Session

In [2]:
@dataclass
class MappingConfig:
    """Configuration for Porto→Casablanca mapping pipeline."""
    # ── Cities ─────────────────────────────────────────────────────────────────
    porto_place: str = "Porto, Portugal"
    casa_place:  str = "Casablanca, Morocco"

    # ── Graph cache (shared with notebook 02 & mapping/ scripts) ──────────────
    porto_graphml: str = "/tmp/porto_drive.graphml"
    casa_graphml:  str = "/tmp/casablanca_drive.graphml"

    # ── Porto data ─────────────────────────────────────────────────────────────
    porto_csv_s3:    str = "s3a://taasim/raw/porto-trips/train.csv"
    porto_csv_local: str = "../../raw/porto-trips/train.csv"

    # ── Zone mapping ──────────────────────────────────────────────────────────
    zones_csv:     str = "../metadata/zone_mapping.csv"
    zones_csv_abs: str = "../../metadata/zone_mapping.csv"

    # ── CBD centroid ───────────────────────────────────────────────────────────
    cbd_lon: float = -7.620
    cbd_lat: float = 33.588

    # ── Profile ────────────────────────────────────────────────────────────────
    trip_limit: int  = 5_000     # Porto trips to map (quick profile)
    workers:    int  = 2         # ProcessPoolExecutor cap (8 GB guardrail)
    h3_res:     int  = 8

    # ── Tariff (Arrêté n° 3-71-19, 2024) ──────────────────────────────────────
    flag_day:       float = 7.00
    flag_night:     float = 10.50
    rate_km_day:    float = 3.50
    rate_km_night:  float = 5.25
    rate_min_day:   float = 0.60
    rate_min_night: float = 0.90
    min_fare:       float = 10.00
    taxi_pool_size: int   = 5_000

    # ── Output ─────────────────────────────────────────────────────────────────
    output_s3:    str = "s3a://taasim/curated/mapped_casa_trips/"
    output_local: str = "./data/mapped_casa_trips/"


CFG = MappingConfig()

# Speed profile (km/h) — Casablanca petit taxi
CASA_SPEED_KMH = np.array([
    42,45,47,47,45,40,  32,18,16,25,30,28,  22,20,28,30,28,15,  14,14,30,35,38,42,
], dtype=float)

POPULATION_2024 = {
    "Sidi Belyout": 138_294,
    "Maarif": 139_669,
    "Anfa": 66_202,
    "Hay Hassani": 537_509,
    "Mers Sultan": 98_139,
    "Ain Chock": 374_707,
    "Hay Mohammadi": 104_232,
    "Sidi Bernoussi": 154_919,
    "Ain Sebaa": 155_828,
    "Roches Noires": 104_775,
    "Sidi Moumen": 551_443,
    "El Fida": 125_933,
    "Mechouar": 2_101,
    "Ben Msik": 105_612,
    "Sbata": 101_640,
    "Moulay Rachid": 475_000,
}

CASA_LANDMARKS = [
    {"name": "Gare Casa-Port", "lat": 33.599181, "lon": -7.611875, "icon": "train"},
    {"name": "Gare Casa-Voyageurs", "lat": 33.589556, "lon": -7.59083, "icon": "train"},
    {"name": "Twin Center", "lat": 33.58611, "lon": -7.63194, "icon": "building"},
    {"name": "Hassan II Mosque", "lat": 33.6085, "lon": -7.6327, "icon": "star"},
    {"name": "Morocco Mall", "lat": 33.575729, "lon": -7.706956, "icon": "shopping-cart"},
    {"name": "CHU Ibn Rochd", "lat": 33.57891, "lon": -7.62155, "icon": "plus-sign"},
    {"name": "Université Hassan II", "lat": 33.5414, "lon": -7.59972, "icon": "education"},
]

spark = (
    SparkSession.builder
    .appName("TaaSim-Mapping-NoteBook-03")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | trip_limit={CFG.trip_limit:,} | workers={CFG.workers}")

Spark 3.5.0 | trip_limit=5,000 | workers=2


## §2 · Load Porto + Casablanca Road Networks

We need **both** graphs:
- **Porto** graph → to compute the source city's bounding box for the affine transform
- **Casablanca** graph → the target road network for snapping and routing

Each graph is cached as `.graphml` to avoid repeated Overpass API calls.
The Casablanca cache at `/tmp/casablanca_drive.graphml` is **shared** with notebook 02.

In [ ]:
def load_graph(place: str, cache_path: str) -> nx.MultiDiGraph:
    """Load cached graph or download from OSM (from porto_to_casa_mapper.py)."""
    p = Path(cache_path)
    if p.exists():
        G = ox.load_graphml(p)
        print(f"  📂 Cached: {p.name}  ({G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges)")
        return G
    print(f"  🌐 Downloading {place}…")
    G = ox.graph_from_place(place, network_type="drive", simplify=True)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    ox.save_graphml(G, p)
    print(f"  💾 Saved: {p.name}  ({G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges)")
    return G


print("[1/2] Loading Porto road network…")
G_porto = load_graph(CFG.porto_place, CFG.porto_graphml)

print("[2/2] Loading Casablanca road network…")
G_casa = load_graph(CFG.casa_place, CFG.casa_graphml)

print("\nBuilding undirected routing graph (avoids one-way blocking)…")
G_undirected = G_casa.to_undirected()
print(f"  ✅ Undirected graph: {G_undirected.number_of_nodes():,} nodes, {G_undirected.number_of_edges():,} edges")

[1/2] Loading Porto road network…
  🌐 Downloading Porto, Portugal…
  💾 Saved: porto_drive.graphml  (5,041 nodes, 10,555 edges)
[2/2] Loading Casablanca road network…
  🌐 Downloading Casablanca, Morocco…


## §3 · Compute Affine Transform: Porto ↔ Casablanca

### The algorithm 

1. Extract the **bounding box** of each city's road graph
2. For any Porto coordinate `(lon, lat)`, compute its **relative position** `[0, 1]` within Porto's bbox
3. Map to the **same relative position** within Casablanca's bbox

```
Porto (30% across in lon)  →  Casablanca (30% across in lon)
```

This preserves the **spatial structure** of Porto trips: a trip from the northwest
to the southeast of Porto becomes a trip from the northwest to the southeast of
Casablanca.

In [ ]:
def compute_transform(G_porto, G_casa):
    """
        Compute the affine mapping from Porto's coordinate space to Casablanca's.
    """
    pn = ox.graph_to_gdfs(G_porto, edges=False)
    cn = ox.graph_to_gdfs(G_casa, edges=False)

    porto_bbox = {
        "min_lon": pn["x"].min(), "max_lon": pn["x"].max(),
        "min_lat": pn["y"].min(), "max_lat": pn["y"].max(),
    }
    casa_bbox = {
        "min_lon": cn["x"].min(), "max_lon": cn["x"].max(),
        "min_lat": cn["y"].min(), "max_lat": cn["y"].max(),
    }

    del pn, cn; gc.collect()

    print(f"  Porto bbox : lon [{porto_bbox['min_lon']:.4f}, {porto_bbox['max_lon']:.4f}]")
    print(f"               lat [{porto_bbox['min_lat']:.4f}, {porto_bbox['max_lat']:.4f}]")
    print(f"  Casa bbox  : lon [{casa_bbox['min_lon']:.4f}, {casa_bbox['max_lon']:.4f}]")
    print(f"               lat [{casa_bbox['min_lat']:.4f}, {casa_bbox['max_lat']:.4f}]")

    return porto_bbox, casa_bbox


def porto_to_casa(lon, lat, porto_bbox, casa_bbox):
    """Map a Porto coordinate to Casablanca using RELATIVE POSITION.
    A point at 30% across Porto maps to 30% across Casablanca.
    """
    rel_lon = (lon - porto_bbox["min_lon"]) / (porto_bbox["max_lon"] - porto_bbox["min_lon"])
    rel_lat = (lat - porto_bbox["min_lat"]) / (porto_bbox["max_lat"] - porto_bbox["min_lat"])
    rel_lon = max(0.0, min(1.0, rel_lon))
    rel_lat = max(0.0, min(1.0, rel_lat))
    casa_lon = casa_bbox["min_lon"] + rel_lon * (casa_bbox["max_lon"] - casa_bbox["min_lon"])
    casa_lat = casa_bbox["min_lat"] + rel_lat * (casa_bbox["max_lat"] - casa_bbox["min_lat"])
    return casa_lon, casa_lat


def porto_to_casa_vec(lons, lats, porto_bbox, casa_bbox):
    """Vectorised version: map arrays of Porto coords to Casablanca."""
    rel_lon = (lons - porto_bbox["min_lon"]) / (porto_bbox["max_lon"] - porto_bbox["min_lon"])
    rel_lat = (lats - porto_bbox["min_lat"]) / (porto_bbox["max_lat"] - porto_bbox["min_lat"])
    rel_lon = np.clip(rel_lon, 0.0, 1.0)
    rel_lat = np.clip(rel_lat, 0.0, 1.0)
    casa_lon = casa_bbox["min_lon"] + rel_lon * (casa_bbox["max_lon"] - casa_bbox["min_lon"])
    casa_lat = casa_bbox["min_lat"] + rel_lat * (casa_bbox["max_lat"] - casa_bbox["min_lat"])
    return casa_lon, casa_lat


print("Computing affine coordinate mapping…")
PORTO_BBOX, CASA_BBOX = compute_transform(G_porto, G_casa)

# Free Porto graph — we don't need it after bbox extraction
del G_porto; gc.collect()
print("\n Affine transform ready. Porto graph freed from memory.")

## §4 · Map Porto Trips to Casablanca Roads

### The core mapping loop 

For each Porto trip:
1. Parse the POLYLINE JSON → list of `[lon, lat]` points
2. Extract START and END points
3. Skip round trips (start ≈ end)
4. Transform START/END to Casablanca coordinates
5. Snap to nearest road nodes (`ox.distance.nearest_nodes`)
6. Find shortest path on **undirected** graph
7. Extract full road geometry (including edge curves)

### Edge geometry extraction 

OSM edges can have a `geometry` attribute — a Shapely `LineString` with the  
actual road curvature. We extract these coordinates in the correct direction  
(forward or reversed) and concatenate them to produce a high-fidelity polyline  
that follows **actual Casablanca road curves, not just straight node-to-node lines**.

In [ ]:
def extract_route_geometry(G, route):
    """Extract full road geometry from a route (list of node IDs).
    Handles edge geometries with curves and direction.
    Returns list of (lon, lat) tuples.
    """
    coords = []
    for i in range(len(route) - 1):
        u, v = route[i], route[i + 1]
        edge_data = G.get_edge_data(u, v)
        reverse = False
        if edge_data is None:
            edge_data = G.get_edge_data(v, u)
            reverse = True
        if edge_data is None:
            if i == 0:
                coords.append((G.nodes[u]["x"], G.nodes[u]["y"]))
            coords.append((G.nodes[v]["x"], G.nodes[v]["y"]))
            continue

        key = list(edge_data.keys())[0]
        edge = edge_data[key]

        if "geometry" in edge:
            geom = list(edge["geometry"].coords)
            if reverse:
                geom = list(reversed(geom))
            else:
                ux, uy = G.nodes[u]["x"], G.nodes[u]["y"]
                if abs(geom[0][0] - ux) + abs(geom[0][1] - uy) > 0.0001:
                    geom = list(reversed(geom))
            start = 0 if i == 0 else 1
            coords.extend(geom[start:])
        else:
            if i == 0:
                coords.append((G.nodes[u]["x"], G.nodes[u]["y"]))
            coords.append((G.nodes[v]["x"], G.nodes[v]["y"]))

    return coords


def map_one_trip(porto_pts, porto_bbox, casa_bbox, G_casa, G_undir):
    """Map one Porto trip to Casablanca's road network.
    Returns (casa_polyline, casa_start, casa_end) or None.
    """
    if not porto_pts or len(porto_pts) < 2:
        return None

    start_lon, start_lat = porto_pts[0]
    end_lon, end_lat = porto_pts[-1]

    # Skip round trips
    if abs(start_lon - end_lon) < 0.001 and abs(start_lat - end_lat) < 0.001:
        return None

    # Transform to Casablanca
    casa_start = porto_to_casa(start_lon, start_lat, porto_bbox, casa_bbox)
    casa_end   = porto_to_casa(end_lon,   end_lat,   porto_bbox, casa_bbox)

    # Snap to nearest road nodes
    origin = ox.distance.nearest_nodes(G_casa, X=casa_start[0], Y=casa_start[1])
    dest   = ox.distance.nearest_nodes(G_casa, X=casa_end[0],   Y=casa_end[1])

    if origin == dest:
        return None

    # Shortest path on UNDIRECTED graph (no one-way blocking)
    try:
        route = nx.shortest_path(G_undir, origin, dest, weight="length")
    except nx.NetworkXNoPath:
        return None

    if len(route) < 2:
        return None

    # Extract full road geometry (including curves)
    coords = extract_route_geometry(G_casa, route)
    if len(coords) < 2:
        return None

    # Compute network distance from edges
    dist_m = 0.0
    for a, b in zip(route[:-1], route[1:]):
        ed = G_casa.get_edge_data(a, b) or G_casa.get_edge_data(b, a)
        if ed:
            k = list(ed.keys())[0]
            dist_m += ed[k].get("length", 0.0)

    return {
        "polyline": [[round(c[0], 6), round(c[1], 6)] for c in coords],
        "origin_lon": coords[0][0], "origin_lat": coords[0][1],
        "dest_lon": coords[-1][0], "dest_lat": coords[-1][1],
        "distance_km": dist_m / 1000.0,
        "n_road_points": len(coords),
        "n_segments": len(route),
    }

In [ ]:
# ── Load Porto trip data ──────────────────────────────────────────────────────
print("Loading Porto trip data…")
porto_pdf = None
for path in [CFG.porto_csv_s3, CFG.porto_csv_local]:
    try:
        sdf = (
            spark.read.option("header", True).csv(path)
            .select("TRIP_ID", "TAXI_ID", "TIMESTAMP", "CALL_TYPE", "DAY_TYPE",
                    "MISSING_DATA", "POLYLINE")
            .filter(F.col("MISSING_DATA") == "False")
            .filter(F.col("POLYLINE").isNotNull())
            .filter(F.col("POLYLINE") != "[]")
            .limit(CFG.trip_limit * 2)  # over-sample, many will be filtered
        )
        porto_pdf = sdf.toPandas()
        print(f" Loaded {len(porto_pdf):,} trips from {path}")
        break
    except Exception as e:
        print(f"  ✗ {path}: {str(e)[:100]}")

if porto_pdf is None:
    raise RuntimeError("Cannot load Porto CSV. Check S3 or local path.")

# ── Map each trip ─────────────────────────────────────────────────────────────
print(f"\nMapping {len(porto_pdf):,} Porto trips to Casablanca roads…")
t0 = time.time()

results = []
ok = skip_short = skip_round = skip_err = 0

for i, row in porto_pdf.iterrows():
    try:
        pts = json.loads(row["POLYLINE"])
        if not pts or len(pts) < 2:
            skip_short += 1
            continue

        result = map_one_trip(pts, PORTO_BBOX, CASA_BBOX, G_casa, G_undirected)

        if result is None:
            skip_round += 1
            continue

        result["trip_id"]        = i + 1
        result["taxi_id"]        = int(row.get("TAXI_ID", 0))
        result["timestamp"]      = int(row.get("TIMESTAMP", 0))
        result["call_type"]      = str(row.get("CALL_TYPE", "B"))
        result["day_type"]       = str(row.get("DAY_TYPE", "A"))
        result["porto_n_points"] = len(pts)
        result["porto_start"]    = pts[0]
        result["porto_end"]      = pts[-1]
        results.append(result)
        ok += 1

    except Exception as e:
        skip_err += 1
        if skip_err <= 3:
            print(f"    ⚠ Trip {i}: {e}")

    if (i + 1) % 500 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        print(f"    {i+1:,}/{len(porto_pdf):,} — {ok:,} ok, {skip_short} short, "
              f"{skip_round} skip, {skip_err} err ({rate:.0f}/sec)")

    if ok >= CFG.trip_limit:
        print(f"    Reached trip_limit={CFG.trip_limit:,}. Stopping.")
        break

elapsed = time.time() - t0
print(f"\n{ok:,} trips mapped in {elapsed:.1f}s ({ok/max(elapsed,1):.0f} trips/sec)")
print(f"   Skipped: {skip_short} short, {skip_round} round/no-path, {skip_err} errors")

mapped_pdf = pd.DataFrame(results)
print(f"\nMapped DataFrame: {len(mapped_pdf):,} rows")
display(mapped_pdf[["trip_id","origin_lon","origin_lat","dest_lon","dest_lat",
                     "distance_km","n_road_points"]].head())

## §5 · Enrichment: Zones, H3, Timestamps, Duration, Fare

Now we have road-snapped Casablanca trips. We enrich each trip with:

1. **Zone assignment** — which arrondissement does the origin/destination fall in?
2. **H3 hex cells** — resolution 8 hexagonal indexing
3. **Casablanca timestamps** — Porto timestamps remapped to 2024 Morocco timezone
4. **Duration** — from distance / speed profile with directional congestion
5. **Fare** — Arrêté n° 3-71-19 tariff

In [ ]:
# ── Load zones — real OSM polygons with bbox fallback ─────────────────────────
def load_zones(cfg):
    for p in [cfg.zones_csv, cfg.zones_csv_abs]:
        if Path(p).exists():
            df = pd.read_csv(p).rename(columns={"arrondissement_id": "zone_id"})
            med_pop = int(np.median(list(POPULATION_2024.values())))
            df["population"] = df["zone_name"].map(POPULATION_2024).fillna(med_pop).astype(int)

            # Try to fetch real OSM administrative polygons for each arrondissement
            real_polys = []
            for _, row in df.iterrows():
                zname = row["zone_name"]
                try:
                    gdf_osm = ox.geocode_to_gdf(f"{zname}, Casablanca, Morocco")
                    real_polys.append(gdf_osm.geometry.iloc[0])
                    print(f"    ✓ {zname}: real OSM polygon")
                except Exception:
                    # Fallback to bounding-box rectangle
                    lo, hi = row["lon_min"], row["lon_max"]
                    la, ha = row["lat_min"], row["lat_max"]
                    real_polys.append(Polygon([(lo,la),(hi,la),(hi,ha),(lo,ha)]))
                    print(f"    ⚠ {zname}: bbox fallback (OSM geocode failed)")

            gdf = gpd.GeoDataFrame(df, geometry=real_polys, crs="EPSG:4326")
            gdf["centroid_lon"] = gdf.geometry.centroid.x
            gdf["centroid_lat"] = gdf.geometry.centroid.y
            return gdf
    raise FileNotFoundError("zone_mapping.csv not found")

zones_gdf = load_zones(CFG)
print(f"Zones: {len(zones_gdf)}")

# ── Assign origin/dest to zones ───────────────────────────────────────────────
origin_pts = gpd.GeoDataFrame(
    mapped_pdf[["trip_id"]],
    geometry=[Point(lon, lat) for lon, lat in zip(mapped_pdf.origin_lon, mapped_pdf.origin_lat)],
    crs="EPSG:4326"
)
dest_pts = gpd.GeoDataFrame(
    mapped_pdf[["trip_id"]],
    geometry=[Point(lon, lat) for lon, lat in zip(mapped_pdf.dest_lon, mapped_pdf.dest_lat)],
    crs="EPSG:4326"
)

# sjoin can produce duplicates when bbox zones overlap — drop_duplicates fixes it
oj = gpd.sjoin(origin_pts, zones_gdf[["zone_id","zone_name","geometry"]],
               predicate="within", how="left")
oj = oj[~oj.index.duplicated(keep="first")]          # keep first zone match
oj = oj.reindex(mapped_pdf.index)                      # align to mapped_pdf
mapped_pdf["origin_zone"]      = oj["zone_id"].values
mapped_pdf["origin_zone_name"] = oj["zone_name"].values

dj = gpd.sjoin(dest_pts, zones_gdf[["zone_id","zone_name","geometry"]],
               predicate="within", how="left")
dj = dj[~dj.index.duplicated(keep="first")]          # keep first zone match
dj = dj.reindex(mapped_pdf.index)                      # align to mapped_pdf
mapped_pdf["destination_zone"]      = dj["zone_id"].values
mapped_pdf["destination_zone_name"] = dj["zone_name"].values

# Fill NaN zones (points at zone boundary)
mapped_pdf["origin_zone"]      = mapped_pdf["origin_zone"].fillna(1).astype(int)
mapped_pdf["destination_zone"] = mapped_pdf["destination_zone"].fillna(1).astype(int)

# ── H3 hex assignment ─────────────────────────────────────────────────────────
if H3_AVAILABLE:
    def _h3(lat, lon, res):
        try: return h3.latlng_to_cell(lat, lon, res)
        except:
            try: return h3.geo_to_h3(lat, lon, res)
            except: return None
    mapped_pdf["h3_origin"] = [_h3(la,lo,CFG.h3_res) for la,lo in zip(mapped_pdf.origin_lat, mapped_pdf.origin_lon)]
    mapped_pdf["h3_dest"]   = [_h3(la,lo,CFG.h3_res) for la,lo in zip(mapped_pdf.dest_lat, mapped_pdf.dest_lon)]
    print(f"H3: {mapped_pdf['h3_origin'].nunique()} origin hexes, {mapped_pdf['h3_dest'].nunique()} dest hexes")
else:
    mapped_pdf["h3_origin"] = None; mapped_pdf["h3_dest"] = None

# ── Timestamps (Porto → Casablanca 2024) ──────────────────────────────────────
mapped_pdf["timestamp"] = pd.to_numeric(mapped_pdf["timestamp"], errors="coerce").fillna(0).astype(np.int64)
# Porto timestamps are 2013-2014. Shift to 2024 by adding ~10 years offset.
PORTO_2014_BASE = int(pd.Timestamp("2014-01-01", tz="UTC").timestamp())
CASA_2024_BASE  = int(pd.Timestamp("2024-01-01", tz="UTC").timestamp())
OFFSET = CASA_2024_BASE - PORTO_2014_BASE
# Keep time-of-day and day-of-week, shift year
mapped_pdf["timestamp"] = mapped_pdf["timestamp"] + OFFSET

# Derived temporal
mapped_pdf["trip_hour"] = (mapped_pdf["timestamp"] % 86_400 // 3_600).astype(int)
ts_arr = mapped_pdf["timestamp"].to_numpy()
_hours = mapped_pdf["trip_hour"].to_numpy()

# ── CBD directionality ────────────────────────────────────────────────────────
def haversine_km_vec(lon1, lat1, lon2, lat2):
    R = 6371.0088
    lon1,lat1,lon2,lat2 = np.radians(lon1),np.radians(lat1),np.radians(lon2),np.radians(lat2)
    dlat=lat2-lat1; dlon=lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))

def bearing_vec(lon1,lat1,lon2,lat2):
    lat1r,lat2r = np.radians(lat1),np.radians(lat2)
    dlonr = np.radians(lon2-lon1)
    x = np.sin(dlonr)*np.cos(lat2r)
    y = np.cos(lat1r)*np.sin(lat2r) - np.sin(lat1r)*np.cos(lat2r)*np.cos(dlonr)
    return (np.degrees(np.arctan2(x,y))+360)%360

olons = mapped_pdf["origin_lon"].to_numpy()
olats = mapped_pdf["origin_lat"].to_numpy()
dlons = mapped_pdf["dest_lon"].to_numpy()
dlats = mapped_pdf["dest_lat"].to_numpy()

b_trip = bearing_vec(olons, olats, dlons, dlats)
b_cbd  = bearing_vec(olons, olats, np.full(len(olons), CFG.cbd_lon), np.full(len(olons), CFG.cbd_lat))
delta  = np.abs(b_trip - b_cbd)
delta  = np.minimum(delta, 360.0 - delta)
mapped_pdf["is_cbd_bound"] = delta < 60.0

# ── Duration with directional congestion ──────────────────────────────────────
speed = CASA_SPEED_KMH[_hours].copy()
am = (_hours >= 7) & (_hours <= 9)
pm = (_hours >= 17) & (_hours <= 20)
cbd = mapped_pdf["is_cbd_bound"].to_numpy()
speed = np.where(am & cbd,  speed*0.75, speed)
speed = np.where(am & ~cbd, speed*1.05, speed)
speed = np.where(pm & ~cbd, speed*0.80, speed)
speed = np.where(pm & cbd,  speed*1.05, speed)
speed = np.maximum(speed, 5.0)

dist_arr = mapped_pdf["distance_km"].to_numpy(float)
raw_s = (dist_arr / speed) * 3600.0
jitter = RNG.normal(1.0, 0.12, size=len(mapped_pdf))
mapped_pdf["duration_sec"] = np.maximum(raw_s * jitter, 60).astype(np.int64)

# ── Fare (Arrêté n° 3-71-19) ──────────────────────────────────────────────────
is_night = (_hours >= 20) | (_hours < 6)
flag  = np.where(is_night, CFG.flag_night, CFG.flag_day)
r_km  = np.where(is_night, CFG.rate_km_night, CFG.rate_km_day)
r_min = np.where(is_night, CFG.rate_min_night, CFG.rate_min_day)
fare  = flag + r_km * dist_arr + r_min * (mapped_pdf["duration_sec"].to_numpy() / 60.0)
mapped_pdf["fare_mad"] = np.round(np.maximum(fare, CFG.min_fare) * 2) / 2

mapped_pdf["taxi_id"] = RNG.integers(1, CFG.taxi_pool_size+1, size=len(mapped_pdf))

print(f"\n📊 Enrichment complete:")
print(f"  Median distance : {mapped_pdf['distance_km'].median():.2f} km")
print(f"  Mean fare       : {mapped_pdf['fare_mad'].mean():.2f} MAD")
print(f"  Mean duration   : {mapped_pdf['duration_sec'].mean()/60:.1f} min")
print(f"  CBD-bound       : {mapped_pdf['is_cbd_bound'].mean():.1%}")

## §6 · Parquet Write — Partitioned by `day_type`

In [ ]:
OUT_COLS = [
    "trip_id", "taxi_id", "timestamp", "day_type", "call_type",
    "origin_zone", "destination_zone", "origin_zone_name", "destination_zone_name",
    "origin_lon", "origin_lat", "dest_lon", "dest_lat",
    "h3_origin", "h3_dest",
    "polyline", "distance_km", "duration_sec", "fare_mad",
    "n_road_points", "is_cbd_bound",
]

final_pdf = mapped_pdf[[c for c in OUT_COLS if c in mapped_pdf.columns]].copy()
final_pdf["polyline"] = final_pdf["polyline"].apply(
    lambda x: json.dumps(x) if isinstance(x, list) else None
)

# Create Spark DF
sdf = spark.createDataFrame(final_pdf).cache()
print(f"Spark DataFrame: {sdf.count():,} rows")

# Write
output_path = CFG.output_s3
try:
    sdf.write.mode("overwrite").partitionBy("day_type").parquet(output_path)
    print(f"💾 Saved to: {output_path}")
except Exception as e:
    output_path = CFG.output_local
    os.makedirs(output_path, exist_ok=True)
    sdf.write.mode("overwrite").partitionBy("day_type").parquet(output_path)
    print(f"💾 Saved locally: {output_path}")

sdf.groupBy("day_type").count().show()

## §7 · Before/After Comparison 

Visualise one trip showing:
- **RED dashed line + dots** = BEFORE — raw affine-transformed points (off-road)
- **GREEN solid line** = AFTER — shortest path on actual Casablanca roads

In [ ]:
# Pick a good trip for comparison (15-50 Porto GPS points)
_good = mapped_pdf[
    (mapped_pdf["porto_n_points"] >= 15) &
    (mapped_pdf["porto_n_points"] <= 50) &
    (mapped_pdf["distance_km"] > 1.0)
]
if len(_good) > 0:
    _row = _good.iloc[0]
else:
    _row = mapped_pdf.iloc[0]

# BEFORE: all raw-transformed points
porto_pts = _row["porto_start"], _row["porto_end"]
_before_raw = []
# Re-parse the original Porto polyline for this trip
_orig_idx = _row["trip_id"] - 1
if _orig_idx < len(porto_pdf):
    _orig_pts = json.loads(porto_pdf.iloc[_orig_idx]["POLYLINE"])
    for p in _orig_pts:
        c = porto_to_casa(p[0], p[1], PORTO_BBOX, CASA_BBOX)
        _before_raw.append(c)

_before_ll = [[b[1], b[0]] for b in _before_raw]

# AFTER: road-snapped polyline
_after = json.loads(_row["polyline"]) if isinstance(_row["polyline"], str) else _row["polyline"]
_after_ll = [[p[1], p[0]] for p in _after]

# Build comparison map
center = [np.mean([b[0] for b in _before_ll]) if _before_ll else 33.57,
          np.mean([b[1] for b in _before_ll]) if _before_ll else -7.59]
m_cmp = folium.Map(location=center, zoom_start=14, tiles="OpenStreetMap")

# RED = BEFORE (raw transformed points — off-road)
if _before_ll and len(_before_ll) >= 2:
    folium.PolyLine(_before_ll, color="red", weight=3, opacity=0.7,
                    dash_array="8", tooltip="BEFORE: Raw points (off-road)").add_to(m_cmp)
    for j, c in enumerate(_before_ll):
        folium.CircleMarker(c, radius=4, color="red", fill=True, fill_opacity=0.8,
                            tooltip=f"Raw #{j}").add_to(m_cmp)

# GREEN = AFTER (road-snapped)
if _after_ll and len(_after_ll) >= 2:
    folium.PolyLine(_after_ll, color="green", weight=5, opacity=0.9,
                    tooltip="AFTER: Road-snapped route").add_to(m_cmp)

# Start/end markers
if _before_ll:
    folium.Marker(_before_ll[0], popup="START",
                  icon=folium.Icon(color="green", icon="play")).add_to(m_cmp)
    folium.Marker(_before_ll[-1], popup="END (raw)",
                  icon=folium.Icon(color="red", icon="stop")).add_to(m_cmp)
if _after_ll:
    folium.Marker(_after_ll[-1], popup="END (road)",
                  icon=folium.Icon(color="darkgreen", icon="flag")).add_to(m_cmp)

# Legend 
legend = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:15px 20px; border-radius:10px;
     border:2px solid #333; font-size:14px; font-family:Arial;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
    <b style="font-size:16px;">🚕 Trip Mapping Comparison</b><br><br>
    <span style="color:red; font-weight:bold;">━ ━ ━ ●</span> BEFORE: Raw transformed points<br>
    <small style="color:#666;">  (off-road, through buildings)</small><br><br>
    <span style="color:green; font-weight:bold;">━━━━━</span> AFTER: Road-snapped route<br>
    <small style="color:#666;">  (follows actual Casablanca streets)</small><br><br>
    <span style="color:green;">▶</span> START &nbsp;
    <span style="color:red;">■</span> END (raw) &nbsp;
    <span style="color:darkgreen;">⚑</span> END (road)
</div>
"""
m_cmp.get_root().html.add_child(folium.Element(legend))

print(f"🔍 Trip {_row['trip_id']}: {len(_before_raw)} raw pts → {_row['n_road_points']} road pts")
print(f"   Distance: {_row['distance_km']:.2f} km | Fare: {_row['fare_mad']:.0f} MAD")
m_cmp

## §8 · Validation Panels (6 charts)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("TaaSim-06 Map-Matching Validation Suite", fontsize=14, fontweight="bold", y=1.01)

# P1: Distance distribution
ax = axes[0,0]
_d = final_pdf["distance_km"].dropna()
_d = _d[(_d > 0.1) & (_d < 40)]
ax.hist(_d, bins=50, color="#f4743b", alpha=0.8, edgecolor="white", lw=0.4)
ax.axvline(_d.median(), color="red", ls="--", lw=2, label=f"Median {_d.median():.1f} km")
ax.set_xlabel("Distance (km)"); ax.set_ylabel("Trips")
ax.set_title("P1 · Trip Distance"); ax.legend(fontsize=8)

# P2: OD flow heatmap
ax = axes[0,1]
if "origin_zone" in final_pdf.columns and "destination_zone" in final_pdf.columns:
    od = pd.crosstab(final_pdf["origin_zone"], final_pdf["destination_zone"])
    im = ax.imshow(od.values, aspect="auto", cmap="YlOrRd")
    ax.set_title("P2 · OD Flow Matrix"); fig.colorbar(im, ax=ax, label="Trips")
    ax.set_xlabel("Dest zone"); ax.set_ylabel("Origin zone")

# P3: Fare distribution
ax = axes[0,2]
_f = final_pdf["fare_mad"].dropna()
_f = _f[_f < 300]
ax.hist(_f, bins=50, color="#1565C0", alpha=0.8, edgecolor="white", lw=0.4)
ax.axvspan(25, 80, alpha=0.1, color="green", label="HACA band [25-80 MAD]")
ax.axvline(_f.mean(), color="red", ls="--", lw=2, label=f"Mean {_f.mean():.1f} MAD")
ax.set_title("P3 · Fare Distribution"); ax.legend(fontsize=8)
ax.set_xlabel("Fare (MAD)"); ax.set_ylabel("Trips")

# P4: Duration
ax = axes[1,0]
_dur = final_pdf["duration_sec"].dropna() / 60
_dur = _dur[_dur < 90]
ax.hist(_dur, bins=50, color="#E65100", alpha=0.8, edgecolor="white", lw=0.4)
ax.axvline(_dur.mean(), color="red", ls="--", lw=2, label=f"Mean {_dur.mean():.1f} min")
ax.set_title("P4 · Duration"); ax.legend(fontsize=8)
ax.set_xlabel("Duration (min)"); ax.set_ylabel("Trips")

# P5: Road points per trip
ax = axes[1,1]
if "n_road_points" in mapped_pdf.columns:
    _rp = mapped_pdf["n_road_points"]
    ax.hist(_rp, bins=50, color="#27ae60", alpha=0.8, edgecolor="white", lw=0.4)
    ax.axvline(_rp.median(), color="red", ls="--", lw=2, label=f"Median {_rp.median():.0f} pts")
    ax.set_title(f"P5 · Road Geometry Points / Trip")
    ax.set_xlabel("Points"); ax.set_ylabel("Trips"); ax.legend(fontsize=8)

# P6: Hour distribution
ax = axes[1,2]
if "trip_hour" in mapped_pdf.columns:
    mapped_pdf["trip_hour"].hist(bins=24, ax=ax, color="#8e44ad", alpha=0.8, edgecolor="white")
    ax.set_title("P6 · Departure Hour"); ax.set_xlabel("Hour"); ax.set_ylabel("Trips")

plt.tight_layout()
plt.savefig("validation_06_mapping.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: validation_06_mapping.png")

## §9 · Interactive Folium Maps

### Map A — Heatmap of trip origins with Casablanca landmarks
### Map B — Sample of road-snapped routes

In [ ]:
# ── Map A: Trip Origins Heatmap + Real Arrondissement Polygons ─────────────────
import branca.colormap as cm

center = [zones_gdf["centroid_lat"].mean(), zones_gdf["centroid_lon"].mean()]
m1 = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")

# Trip-origin heatmap layer
heat_data = mapped_pdf[["origin_lat","origin_lon"]].dropna().sample(
    n=min(15_000, len(mapped_pdf)), random_state=42).values.tolist()
HeatMap(heat_data, radius=10, blur=8, min_opacity=0.3, name="Trip Origins Heatmap").add_to(m1)

# Real arrondissement polygon boundaries (not rectangles!)
_zone_colors = [
    "#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6","#1abc9c",
    "#e67e22","#2980b9","#27ae60","#c0392b","#8e44ad","#16a085",
    "#d35400","#2c3e50","#f1c40f","#7f8c8d",
]
for idx, (_, r) in enumerate(zones_gdf.iterrows()):
    color = _zone_colors[idx % len(_zone_colors)]
    folium.GeoJson(
        r.geometry.__geo_interface__,
        style_function=lambda x, c=color: {
            "fillOpacity": 0.12,
            "fillColor": c,
            "color": c,
            "weight": 2.5,
            "dashArray": "",
        },
        tooltip=folium.Tooltip(
            f"<b>{r['zone_name']}</b><br>"
            f"Population: {r['population']:,}<br>"
            f"Zone ID: {r['zone_id']}",
            sticky=True,
        ),
        name=f"Zone: {r['zone_name']}",
    ).add_to(m1)
    # Zone label at centroid
    folium.Marker(
        location=[r["centroid_lat"], r["centroid_lon"]],
        icon=folium.DivIcon(html=f'''
            <div style="font-size:9px; font-weight:bold; color:{color};
                        text-shadow:1px 1px 2px white, -1px -1px 2px white;
                        white-space:nowrap;">
                {r["zone_name"]}
            </div>'''),
    ).add_to(m1)

# Landmarks
for lm in CASA_LANDMARKS:
    folium.Marker([lm["lat"],lm["lon"]], tooltip=lm["name"],
                  icon=folium.Icon(color="red", icon=lm["icon"], prefix="glyphicon")).add_to(m1)

folium.LayerControl(collapsed=False).add_to(m1)
print(f"🗺️  Heatmap: {len(heat_data):,} origins | {len(zones_gdf)} arrondissement polygons")
m1

In [ ]:
# ── Map B: Road-snapped route polylines (from visualize.py pattern) ───────────
TRIP_COLORS = [
    "#e6194b","#3cb44b","#4363d8","#f58231","#911eb4",
    "#42d4f4","#f032e6","#bfef45","#fabed4","#469990",
    "#dcbeff","#9A6324","#800000","#aaffc3","#808000","#000075",
]

_n_show = min(100, len(mapped_pdf))
_sample = mapped_pdf.sample(_n_show, random_state=42)

m2 = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")

for idx, (_, row) in enumerate(_sample.iterrows()):
    try:
        _pl = json.loads(row["polyline"]) if isinstance(row["polyline"], str) else row["polyline"]
        if not _pl or len(_pl) < 2: continue
        coords_ll = [[p[1], p[0]] for p in _pl]
        color = TRIP_COLORS[idx % len(TRIP_COLORS)]
        folium.PolyLine(coords_ll, weight=2.5, opacity=0.7, color=color,
                        tooltip=f"Trip {row['trip_id']} | {row['distance_km']:.1f}km | {row['fare_mad']:.0f} MAD"
        ).add_to(m2)
        folium.CircleMarker(coords_ll[0], radius=4, color="green", fill=True).add_to(m2)
        folium.CircleMarker(coords_ll[-1], radius=4, color="red", fill=True).add_to(m2)
    except: pass

for lm in CASA_LANDMARKS:
    folium.Marker([lm["lat"],lm["lon"]], tooltip=lm["name"],
                  icon=folium.Icon(color="orange", icon=lm["icon"], prefix="glyphicon")).add_to(m2)

print(f"🛣️  {_n_show} road-snapped trips on Casablanca streets")
m2

## §9b · Population Choropleth — Arrondissement Density Heatmap

Each arrondissement polygon is **colored by population** (RGPH-2024).
Population numbers are displayed directly on each zone.

In [ ]:
# ── Population Choropleth Heatmap per Arrondissement ─────────────────────────
import branca.colormap as cm

center = [zones_gdf["centroid_lat"].mean(), zones_gdf["centroid_lon"].mean()]
m_pop = folium.Map(location=center, zoom_start=12, tiles="CartoDB dark_matter")

# Build color scale from population
pop_min = zones_gdf["population"].min()
pop_max = zones_gdf["population"].max()
colormap = cm.LinearColormap(
    colors=["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"],
    vmin=pop_min, vmax=pop_max,
    caption="Population (RGPH-2024)",
)
colormap.add_to(m_pop)

for _, r in zones_gdf.iterrows():
    pop = r["population"]
    color = colormap(pop)

    # Arrondissement polygon — filled by population color
    folium.GeoJson(
        r.geometry.__geo_interface__,
        style_function=lambda x, c=color: {
            "fillColor": c,
            "fillOpacity": 0.65,
            "color": "white",
            "weight": 2,
        },
        tooltip=folium.Tooltip(
            f"<b>{r['zone_name']}</b><br>"
            f"Population: <b>{pop:,}</b><br>"
            f"Zone ID: {r['zone_id']}",
            sticky=True,
        ),
    ).add_to(m_pop)

    # Population number label at centroid
    folium.Marker(
        location=[r["centroid_lat"], r["centroid_lon"]],
        icon=folium.DivIcon(html=f'''
            <div style="font-size:11px; font-weight:bold; color:white;
                        text-shadow:1px 1px 3px black, -1px -1px 3px black;
                        text-align:center; white-space:nowrap;">
                {r["zone_name"]}<br>
                <span style="font-size:13px;">{pop:,}</span>
            </div>'''),
    ).add_to(m_pop)

# Landmarks for reference
for lm in CASA_LANDMARKS:
    folium.Marker(
        [lm["lat"], lm["lon"]], tooltip=lm["name"],
        icon=folium.Icon(color="lightgray", icon=lm["icon"], prefix="glyphicon"),
    ).add_to(m_pop)

print(f"🏙️  Population Choropleth: {len(zones_gdf)} arrondissements")
print(f"   Range: {pop_min:,} (Mechouar) → {pop_max:,} (Hay Hassani)")
display(zones_gdf[["zone_name", "population"]].sort_values("population", ascending=False).reset_index(drop=True))
m_pop


## §10 · Statistical Validation Suite

In [ ]:
_PASS = "✅ PASS"; _FAIL = "⚠️  FAIL"
_results = []

def _check(label, cond, detail=""):
    s = _PASS if cond else _FAIL
    print(f"  {s}  {label}   [{detail}]")
    _results.append(cond)

print("="*64)
print("§10  TaaSim-06 · Mapping Pipeline Validation Suite")
print("="*64)

_d = mapped_pdf["distance_km"].dropna()
_f = mapped_pdf["fare_mad"].dropna()
_dur = mapped_pdf["duration_sec"].dropna() / 60

print("\n  Distance Realism")
_check("Median distance ∈ [1.0, 10.0] km", 1.0 <= _d.median() <= 10.0, f"median={_d.median():.2f} km")
_check("Mean distance ∈ [1.5, 10.0] km", 1.5 <= _d.mean() <= 10.0, f"mean={_d.mean():.2f} km")
_check("95th-pct < 25 km", _d.quantile(0.95) < 25, f"p95={_d.quantile(0.95):.2f} km")

print("\n  Fare Realism")
_check("Mean fare ∈ [20, 150] MAD", 20 <= _f.mean() <= 150, f"mean={_f.mean():.1f} MAD")

print("\n  Duration Realism")
_check("Mean duration ∈ [3, 45] min", 3 <= _dur.mean() <= 45, f"mean={_dur.mean():.1f} min")

print("\n  Mapping Quality")
_check(f"Mapped trips > 0", len(mapped_pdf) > 0, f"{len(mapped_pdf):,} trips")
if "n_road_points" in mapped_pdf.columns:
    _check("Median road points ≥ 5", mapped_pdf["n_road_points"].median() >= 5,
           f"median={mapped_pdf['n_road_points'].median():.0f}")

if "origin_zone" in mapped_pdf.columns:
    _nz = mapped_pdf["origin_zone"].nunique()
    _check("Origin zones ≥ 10", _nz >= 10, f"{_nz} zones")

if H3_AVAILABLE and "h3_origin" in mapped_pdf.columns:
    _nh = mapped_pdf["h3_origin"].nunique()
    print(f"\n  H3 hexes: {_nh} origin, {mapped_pdf['h3_dest'].nunique()} dest")

_n_pass = sum(_results)
print(f"\n{'='*64}")
print(f"  RESULT: {_n_pass}/{len(_results)} checks passed")
print(f"  Total mapped trips: {len(mapped_pdf):,}")
print(f"  Output: {output_path}")
print(f"{'='*64}")

## §11 · Pipeline Summary

### What this notebook produced

| Output | Description |
|---|---|
| **Parquet store** | Mapped trips partitioned by `day_type` |
| **Polylines** | JSON-encoded road-following coordinates from OSM edge geometry |
| **H3 hex IDs** | `h3_origin` / `h3_dest` at resolution 8 |
| **Before/After map** | Folium comparison showing raw vs road-snapped |
| **Validation panels** | 6 statistical charts + formal assertion suite |

### Output schema

| Column | Type | Description |
|---|---|---|
| `trip_id` | int | Unique identifier |
| `taxi_id` | int | Fleet index |
| `timestamp` | long | UTC Unix (2024) |
| `day_type` | string | A=weekday, B=weekend |
| `origin/dest_lon/lat` | double | Road-snapped GPS |
| `origin/destination_zone` | int | Arrondissement ID |
| `h3_origin/h3_dest` | string | H3 res-8 cell |
| `polyline` | string | JSON road geometry |
| `distance_km` | double | OSM network distance |
| `duration_sec` | long | With directional congestion |
| `fare_mad` | double | Arrêté n° 3-71-19 |
| `n_road_points` | int | Geometry fidelity |
| `is_cbd_bound` | bool | Trip towards CBD |

---

> **References**  
> - HACA (2019). Enquête d'Opinions sur les Transports Urbains au Maroc.  
> - Arrêté n° 3-71-19 (2024). Tarifs des petits taxis au Maroc.